In [1]:
# Set up all required functions
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql import functions as F
from pyspark.sql.functions import dense_rank, desc,when
from pyspark.sql.functions import dense_rank, desc
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,CharType,ArrayType
from itertools import chain

In [2]:
# Initialize system configurations

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.8-hotspot"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ["PATH"]

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = os.path.join(os.environ["HADOOP_HOME"], "bin") + os.pathsep + os.environ["PATH"]

# Verify BEFORE creating the session

print("PYSPARK_PYTHON:", os.environ.get("PYSPARK_PYTHON"))
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))

spark = SparkSession.builder \
    .appName("pyspark_ylep") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.sql.autoBroadcastJoinThreshold", "64MB") \
    .config("spark.local.dir", "D:/spark_tmp") \
    .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true") \
    .config("spark.python.worker.faulthandler.enabled", "true") \
    .config("spark.sql.execution.pythonUDF.arrow.enabled", "false") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1")\
    .getOrCreate()



PYSPARK_PYTHON: c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\python.exe
HADOOP_HOME: C:\hadoop


c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
# Load data from dim_subcategory
df_dim_subcategory = spark.read.parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_subcategory")

In [4]:
# Load data from dim_category
df_dim_category = spark.read.parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_category")

In [5]:
# Read raw business data
df_read_business_data = spark.read.json("D:/A_Data_engineer/Project_Yelp/Yelp JSON/yelp_academic_dataset_business.json")

In [6]:
# Extract the 'categories' column from business data
df_categories = df_read_business_data.select(F.col("categories"))

In [ ]:
# This function is used to get the category name and subcategory names from the 'categories' column
def get_category_name(text):
    if text is None:
        return "Not Specified", ["Not Specified"]
    
    list_of_element = text.split(",")
    category_name = list_of_element[0].strip()

    del list_of_element[0]
    if len(list_of_element) == 0:
        return category_name, ["Not Specified"]
    
    list_of_element_without_space = [st.lower().strip() for st in list_of_element]

    return category_name, list_of_element_without_space

# Define DataFrame schema
schema = StructType([
    StructField("Categorie_name", StringType(), True),
    StructField("subcategory_names", ArrayType(StringType()), True),
])

# Initialize the UDF
get_category_name_udf = F.udf(get_category_name, schema)

# Step 1: apply UDF, extract fields, then CACHE + materialize 

df_udf = (
    df_categories
    .withColumn("new_eilte", get_category_name_udf(F.col("categories")))
    .withColumn("Categorie_name", F.col("new_eilte.Categorie_name"))
    .withColumn("subcategory_names_list", F.col("new_eilte.subcategory_names"))
    .drop("new_eilte", "categories")
    .cache()
)
df_udf.count()  # Materialization happens at this stage

# Step 2: explode AFTER the UDF stage is fully materialized

df_category_subcategory_names = (
    df_udf
    .withColumn("subcategory_names", F.explode(F.col("subcategory_names_list")))
    .drop("subcategory_names_list")
)
df_category_subcategory_names.show(20, truncate=False)

In [ ]:
#df_category_subcategory_names.filter(F.col("subcategory_names").isin("accessories")).count()

In [ ]:
# Join df_category_subcategory_names with df_dim_category to get category surrogate key
df_join_get_category_sk = df_category_subcategory_names.join(df_dim_category,on = df_dim_category.Category_name == df_category_subcategory_names.Categorie_name ,how= "inner" )

In [ ]:
# Join df_join_get_category_sk  with df_dim_subcategory to get subcategory surrogate key

df_join_get_subcategory_sk = df_join_get_category_sk.join(df_dim_subcategory,on = df_join_get_category_sk.subcategory_names == df_dim_subcategory.Subcategory_name ,how= "inner" )

In [ ]:
# Select only category_pk and category_sk

df_category_subcategory_bridge = df_join_get_subcategory_sk.select(F.col("Category_pk_sk") , F.col("Subcategory_pk_sk") )

In [ ]:
# Write data to the disk
# df_category_subcategory_bridge.write.mode("overwrite").parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_category_subcategory_bridge")

In [ ]:
#df_join_get_category_sk_1.filter(F.col("Categorie_name").isin("Doctors")).count()
#df_join_get_category_sk_1.filter(F.col("tag_name").isin("accessories")).count()
#df_join_get_category_sk.filter(F.col("Categorie_name").isin("Doctors")).count()

2807